In [65]:
import zipfile
import os
import datetime
import pandas as pd

DATA_DIR = "data"
DIVVY_DIR = f"{DATA_DIR}/divvy"

In [8]:
def extractFromZip(dir, file):
    with zipfile.ZipFile(f"{dir}/{file}", mode="r") as zf:
        with zf.open(file.split(".zip")[0]) as f:
            return pd.read_csv(f)

def buildDivvyDataset(dir):
    files = os.listdir(dir)
    df = pd.DataFrame()
    for f in files:
        df = pd.concat([df, extractFromZip(dir, f)])
    return df

In [60]:
divvy_df = buildDivvyDataset(DIVVY_DIR)
weather_df = pd.read_csv(DATA_DIR + "/2025-chicago-weather.csv", skiprows=2)

In [61]:
weather_df["time"] = pd.to_datetime(weather_df["time"])
weather_df["date"] = weather_df["time"].dt.date

daily_weather = (
    weather_df.groupby("date")
    .agg(
        avg_temp=("temperature_2m (°C)", "mean"),
        max_temp=("temperature_2m (°C)", "max"),
        min_temp=("temperature_2m (°C)", "min"),
        cumulative_rain=("rain (mm)", "sum"),
        cumulative_snow=("snowfall (cm)", "sum"),
    )
    .reset_index()
)

In [55]:
divvy_aggs = divvy.groupby([divvy['started_at'].dt.date, divvy['rideable_type'], divvy['start_station_name']]).size()

In [66]:
ridership_df = divvy_aggs.reset_index(name="ride_count")
ridership_df = ridership_df[
    ridership_df["started_at"] >= datetime.date(2025, 1, 1)
]
ridership_df["started_at"] = pd.to_datetime(ridership_df["started_at"]).dt.date
df = pd.merge(
    ridership_df,
    daily_weather,
    left_on="started_at",
    right_on="date",
    how="left",
).drop(columns=["date"])

In [67]:
df

,started_at,rideable_type,start_station_name,ride_count,avg_temp,max_temp,min_temp,cumulative_rain,cumulative_snow
0,2025-01-01,classic_bike,2112 W Peterson Ave,1,-1.329167,0.8,-3.1,0.0,0.07
1,2025-01-01,classic_bike,900 W Harrison St,7,-1.329167,0.8,-3.1,0.0,0.07
2,2025-01-01,classic_bike,Aberdeen St & Jackson Blvd,7,-1.329167,0.8,-3.1,0.0,0.07
3,2025-01-01,classic_bike,Aberdeen St & Monroe St,1,-1.329167,0.8,-3.1,0.0,0.07
4,2025-01-01,classic_bike,Aberdeen St & Randolph St,1,-1.329167,0.8,-3.1,0.0,0.07
...,...,...,...,...,...,...,...,...,...
427513,2025-12-31,electric_bike,Wood St & Chicago Ave,2,-1.445833,1.2,-7.1,0.9,0.98
427514,2025-12-31,electric_bike,Wood St & Taylor St (Temp),1,-1.445833,1.2,-7.1,0.9,0.98
427515,2025-12-31,electric_bike,Woodlawn Ave & 55th St,1,-1.445833,1.2,-7.1,0.9,0.98
427516,2025-12-31,electric_bike,Woodlawn Ave & 58th St,4,-1.445833,1.2,-7.1,0.9,0.98


In [68]:
df.to_csv('data/daily_ridership_with_weather.csv')